# ADA 4.5 · Hybrid sequence model and causality audit

This notebook runs one tiny cycle of the historical ADA 4.5 model,
reproduces its prefix-invariance failure, and trains an explicitly modified
causal variant on a synthetic next-token task.

**Sources.** The historical definitions below are taken from the ARcHi
workspace's Ada_V4.5/ada_v4_5.py at SHA-256 d6e468b1b91029d7c3e4cbb926ca28a33d3a441e0cb6b494592bbcb4fcd6a0be. The internal research
manuscript *ADA: Adaptive Depth Architecture* (July 2026, sections on
topology, GQA, memory and MTP) describes the two SSM layers, learned
memory bank and attention/MoE block. [GQA](https://arxiv.org/abs/2305.13245),
[RoPE](https://arxiv.org/abs/2104.09864), and
[Mamba-2 / SSD](https://arxiv.org/abs/2405.21060) provide background.

The embedded definitions preserve the historical behavior. The second
model uses full SSM scans with token-local residual gating and a prefix-only
attention head gate. A learned slot bank is shared model state. This
experiment does not test online document memory, hardware sparsity,
long-context quality, or the preprint's scaling estimates.

### Run environment

Use Python 3.11 or 3.12 with Jupyter and a CPU PyTorch build (2.6 or later).
Select that environment as the kernel and choose **Restart Kernel and Run All**.
For a fresh environment, install Python packages with
**python -m pip install jupyterlab torch**, selecting the
[CPU wheels](https://pytorch.org/get-started/locally/) if desired.
The cells use synthetic tensors and make no network requests.

In [1]:
from io import BytesIO
import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from types import SimpleNamespace

import torch
from torch import Tensor, nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(2026)
torch.use_deterministic_algorithms(True)
print("PyTorch", torch.__version__, "| device cpu | seed 2026")

PyTorch 2.11.0+cpu | device cpu | seed 2026


## Historical components

The SSM applies a causal depthwise convolution and a readable sequential
state recurrence. Memory addresses learned slots with cosine similarity.
GQA shares key/value heads, computes dense attention, then masks query heads.
The MoE uses top-k routing weights with a separate expert-load signal.
Historical MoD chooses token positions across the full sequence.

In [2]:
@dataclass
class ADAConfig:
    # ── vocabulary ────────────────────────────────────────────────────────────
    vocab_size: int        = 50_272       # GPT-NeoX / cl100k

    # ── core dimensions ───────────────────────────────────────────────────────
    d_model: int           = 2_048
    n_heads: int           = 16
    n_kv_heads: int        = 4            # GQA: n_heads // n_kv_heads = 4:1
    n_layers: int          = 32

    # ── Mamba-2 SSM ───────────────────────────────────────────────────────────
    mamba_d_state: int     = 128          # N: state dimension per SSM head
    mamba_d_conv: int      = 4            # causal depthwise conv kernel
    mamba_expand: int      = 2            # d_inner = expand × d_model
    mamba_n_heads: int     = 64           # H: SSM heads; head_dim = d_inner / H
    mamba_dt_min: float    = 0.001
    mamba_dt_max: float    = 0.1

    # ── Continuous External Memory ────────────────────────────────────────────
    mem_size: int          = 2_048        # number of learnable memory slots
    d_mem: int             = 256          # memory key/value bottleneck dim
    mem_temp: float        = 10.0         # cosine similarity temperature

    # ── Dynamic Sparse GQA ────────────────────────────────────────────────────
    sparse_keep: float     = 0.75         # fraction of heads kept per forward pass

    # ── Fine-Grained MoE ─────────────────────────────────────────────────────
    n_experts: int         = 64
    n_active: int          = 6            # top-k per token
    moe_d_ff: int          = 1_408        # per-expert SwiGLU intermediate dim
    bias_lr: float         = 1e-3         # load-balance bias update rate

    # ── Mixture of Depths ─────────────────────────────────────────────────────
    mod_capacity: float    = 0.5          # fraction of tokens through each Mamba layer

    # ── Multi-Token Prediction ────────────────────────────────────────────────
    n_mtp: int             = 4            # heads predict t+2, t+3, t+4, t+5

    # ── misc ──────────────────────────────────────────────────────────────────
    max_seq_len: int       = 4_096
    dropout: float         = 0.0
    rope_theta: float      = 500_000.0   # LLaMA-3 long-context θ
    norm_eps: float        = 1e-6

    # ── derived ───────────────────────────────────────────────────────────────
    @property
    def head_dim(self) -> int:
        assert self.d_model % self.n_heads == 0
        return self.d_model // self.n_heads

    @property
    def mamba_inner(self) -> int:
        return self.mamba_expand * self.d_model

    @property
    def mamba_head_dim(self) -> int:
        di = self.mamba_inner
        assert di % self.mamba_n_heads == 0, (
            f"mamba_n_heads={self.mamba_n_heads} must divide mamba_inner={di}"
        )
        return di // self.mamba_n_heads

    @property
    def block_pattern(self) -> List[str]:
        """
        4-block cycle: 2× Mamba(MoD), 1× ContinuousMemory, 1× AgentAttn.
        Gives each attention layer two upstream SSM layers + a memory refresh.
        """
        base = ["mamba", "mamba", "memory", "agent_attn"]
        return (base * (self.n_layers // 4 + 1))[: self.n_layers]


class RMSNorm(nn.Module):
    """Root-mean-square normalisation. No additive bias."""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        fp32 = x.float()
        rms  = fp32.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (fp32 * rms).to(x.dtype) * self.weight


def build_rope_cache(
    seq_len: int,
    head_dim: int,
    theta: float = 10_000.0,
    device: Optional[torch.device] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Returns (cos, sin) each (seq_len, head_dim // 2), float32."""
    half     = head_dim // 2
    inv_freq = 1.0 / (theta ** (torch.arange(0, half, device=device).float() / half))
    t        = torch.arange(seq_len, device=device, dtype=torch.float32)
    freqs    = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(
    x: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
    position_ids: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """
    x            : (B, n_heads, T, head_dim)
    cos, sin     : (max_seq, head_dim // 2)
    position_ids : (B, T) — supports non-contiguous positions (KV-cache, MoD)
    """
    T    = x.shape[2]
    half = x.shape[-1] // 2

    if position_ids is not None:
        c = cos[position_ids].unsqueeze(1)     # (B, 1, T, half)
        s = sin[position_ids].unsqueeze(1)
    else:
        c = cos[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, half)
        s = sin[:T].unsqueeze(0).unsqueeze(0)

    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * c - x2 * s, x2 * c + x1 * s], dim=-1)


class SwiGLU(nn.Module):
    """SwiGLU: out = W_down(SiLU(W_gate(x)) ⊙ W_up(x))."""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up   = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff,    d_model, bias=False)
        self.drop   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_down(self.drop(F.silu(self.w_gate(x)) * self.w_up(x)))


class ContinuousMemory(nn.Module):
    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.temp = cfg.mem_temp

        # Learnable memory slots
        self.K = nn.Parameter(torch.randn(cfg.mem_size, cfg.d_mem) * 0.02)
        self.V = nn.Parameter(torch.randn(cfg.mem_size, cfg.d_mem) * 0.02)

        # Projections
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_mem, bias=False)
        self.v_proj = nn.Linear(cfg.d_mem,   cfg.d_model, bias=False)

        # Pre-norm: applied before retrieval; residual added after
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        h = self.norm(x)

        q   = self.q_proj(h)                                     # (B, T, d_mem)
        q_n = F.normalize(q,      p=2, dim=-1)                   # unit vectors
        k_n = F.normalize(self.K, p=2, dim=-1)                   # (mem_size, d_mem)

        sim  = torch.einsum("btd,md->btm", q_n, k_n)            # (B, T, mem_size)
        attn = F.softmax(sim * self.temp, dim=-1)

        read = self.v_proj(attn @ self.V)                        # (B, T, d_model)
        return residual + read


class DynamicSparseGQA(nn.Module):
    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.n_heads   = cfg.n_heads
        self.n_kv      = cfg.n_kv_heads
        self.head_dim  = cfg.head_dim
        self.n_rep     = cfg.n_heads // cfg.n_kv_heads
        self.scale     = cfg.head_dim ** -0.5
        self.k_keep    = max(1, int(cfg.n_heads * cfg.sparse_keep))

        D, H, Hkv, hd = cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.head_dim
        self.wq  = nn.Linear(D, H   * hd, bias=False)
        self.wk  = nn.Linear(D, Hkv * hd, bias=False)
        self.wv  = nn.Linear(D, Hkv * hd, bias=False)
        self.wo  = nn.Linear(H * hd, D,   bias=False)
        self.drop = nn.Dropout(cfg.dropout)

        # Head gate: pools the sequence, outputs one scalar per query head.
        # Separate Linear + sigmoid (not nn.Sequential) to avoid module-level
        # sigmoid() being treated as a Module call with incorrect forward sig.
        self.head_gate_proj = nn.Linear(D, H, bias=False)

        cos, sin = build_rope_cache(cfg.max_seq_len, cfg.head_dim, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(
        self,
        x: torch.Tensor,
        position_ids: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        B, T, _ = x.shape
        hd = self.head_dim

        q = self.wq(x).reshape(B, T, self.n_heads, hd).transpose(1, 2)  # (B,H,T,hd)
        k = self.wk(x).reshape(B, T, self.n_kv,   hd).transpose(1, 2)
        v = self.wv(x).reshape(B, T, self.n_kv,   hd).transpose(1, 2)

        q = apply_rope(q, self.rope_cos, self.rope_sin, position_ids)
        k = apply_rope(k, self.rope_cos, self.rope_sin, position_ids)

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # ── dynamic head gating ───────────────────────────────────────────────
        # Mean-pool the sequence to get a single context vector per sample.
        # Gate projects to (B, n_heads); sigmoid squashes to (0, 1).
        gate_scores  = torch.sigmoid(self.head_gate_proj(x.mean(dim=1)))  # (B, H)
        topk_vals, topk_idx = torch.topk(gate_scores, self.k_keep, dim=-1)

        # Build sparse mask: gate_val at selected heads, 0 elsewhere.
        # Gradient flows through topk_vals (not through topk_idx — same trick as MoE).
        head_mask = torch.zeros_like(gate_scores)                         # (B, H)
        head_mask.scatter_(1, topk_idx, topk_vals)
        head_mask = head_mask[:, :, None, None]                           # (B, H, 1, 1)

        # ── standard scaled dot-product attention ─────────────────────────────
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale       # (B,H,T,T)
        causal = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~causal[None, None], float("-inf"))

        attn = F.softmax(scores.float(), dim=-1).to(x.dtype)
        attn = self.drop(attn)

        # Apply head mask: zero out pruned heads; weight active heads by gate.
        out = (torch.matmul(attn, v) * head_mask)                        # (B,H,T,hd)
        out = out.transpose(1, 2).reshape(B, T, -1)
        return self.wo(out)


class Expert(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up   = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff,    d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class FineGrainedMoE(nn.Module):
    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.n_experts = cfg.n_experts
        self.n_active  = cfg.n_active
        self.bias_lr   = cfg.bias_lr

        self.router  = nn.Linear(cfg.d_model, cfg.n_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(cfg.d_model, cfg.moe_d_ff) for _ in range(cfg.n_experts)
        ])
        # Buffer (not Parameter): excluded from optimizer; updated manually.
        self.register_buffer("expert_bias", torch.zeros(cfg.n_experts))

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, D  = x.shape
        n_tok    = B * T
        x_flat   = x.reshape(n_tok, D)

        logits   = self.router(x_flat)                           # (n_tok, E)
        biased   = logits + self.expert_bias                     # routing signal

        _, topk_idx = torch.topk(biased, self.n_active, dim=-1) # (n_tok, k)

        # Weights from unbiased logits — gradient flows here, not through bias
        weights  = F.softmax(logits.gather(1, topk_idx).float(), dim=-1).to(x.dtype)

        # Dense dispatch: all experts run; pruned by combine weights
        expert_outs = torch.stack(
            [exp(x_flat) for exp in self.experts], dim=1
        )                                                         # (n_tok, E, D)

        combine  = torch.zeros(n_tok, self.n_experts, device=x.device, dtype=x.dtype)
        combine.scatter_(1, topk_idx, weights)
        out      = torch.einsum("te,ted->td", combine, expert_outs)  # (n_tok, D)

        # Online bias update (training only; no loss term; no gradient)
        load     = torch.zeros(self.n_experts, device=x.device, dtype=torch.float32)
        if self.training:
            with torch.no_grad():
                load.scatter_add_(
                    0,
                    topk_idx.flatten(),
                    torch.ones(n_tok * self.n_active, device=x.device),
                )
                load_frac = load / max(1, n_tok * self.n_active)
                self.expert_bias.add_(
                    self.bias_lr * (1.0 / self.n_experts - load_frac)
                )
                load = load_frac
        else:
            load = combine.detach().sum(0)

        return out.reshape(B, T, D), load


class Mamba2SSM(nn.Module):
    def __init__(self, cfg: ADAConfig):
        super().__init__()
        di = cfg.mamba_inner
        N  = cfg.mamba_d_state
        H  = cfg.mamba_n_heads
        hd = cfg.mamba_head_dim

        self.n_heads  = H
        self.head_dim = hd
        self.d_state  = N
        self.d_inner  = di

        # Fused projection: z(di) | x(di) | B(N) | C(N) | dt(H)
        self.in_proj = nn.Linear(cfg.d_model, 2 * di + 2 * N + H, bias=False)

        # Causal depthwise conv — padding d_conv-1 both sides; trim right in forward
        self.conv1d  = nn.Conv1d(
            di, di,
            kernel_size=cfg.mamba_d_conv,
            groups=di,
            padding=cfg.mamba_d_conv - 1,
            bias=True,
        )

        log_dt = (
            torch.rand(H) * (math.log(cfg.mamba_dt_max) - math.log(cfg.mamba_dt_min))
            + math.log(cfg.mamba_dt_min)
        )
        self.dt_bias = nn.Parameter(log_dt)
        self.A_log   = nn.Parameter(
            torch.log(torch.arange(1, H + 1, dtype=torch.float32))
        )
        self.D       = nn.Parameter(torch.ones(H))

        self.out_norm = RMSNorm(di, cfg.norm_eps)
        self.out_proj = nn.Linear(di, cfg.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, _ = x.shape
        H, hd, N, di = self.n_heads, self.head_dim, self.d_state, self.d_inner

        proj = self.in_proj(x)                                   # (B,L,2di+2N+H)
        z, x_in, B_in, C_in, dt = proj.split([di, di, N, N, H], dim=-1)

        x_in = F.silu(
            self.conv1d(x_in.transpose(1, 2))[..., :L]
        ).transpose(1, 2)                                         # (B,L,di)
        x_h  = x_in.reshape(B, L, H, hd)                        # (B,L,H,hd)

        A    = -torch.exp(self.A_log.float())                    # (H,)
        dt   = F.softplus(dt.float() + self.dt_bias)            # (B,L,H)
        Abar = torch.exp(dt * A)                                 # (B,L,H)
        Bbar = dt.unsqueeze(-1) * B_in.float().unsqueeze(2)     # (B,L,H,N)

        h  = x.new_zeros(B, H, hd, N, dtype=torch.float32)
        ys = x.new_empty(B, L, H, hd)

        for t in range(L):
            h        = Abar[:, t, :, None, None] * h + Bbar[:, t, :, None, :] * x_h[:, t, :, :, None].float()
            c_t      = C_in[:, t, None, None, :].float()
            y_skip   = self.D[None, :, None] * x_h[:, t].float()
            ys[:, t] = ((h * c_t).sum(-1) + y_skip).to(x.dtype)

        y = self.out_norm(ys.reshape(B, L, di))
        y = y * F.silu(z)
        return self.out_proj(y)


class Mamba2Block(nn.Module):
    """
    Pre-norm Mamba-2. No residual — MoD wrapper owns the weighted residual.
    Returns the SSM contribution only: SSM(norm(x)).
    """

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ssm  = Mamba2SSM(cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.ssm(self.norm(x))


class MoD(nn.Module):
    def __init__(self, block: nn.Module, d_model: int, capacity: float):
        super().__init__()
        self.block    = block
        self.capacity = capacity
        self.router   = nn.Linear(d_model, 1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        k = max(1, int(T * self.capacity))

        scores = self.router(x).squeeze(-1)                      # (B, T)
        _, top_idx = torch.topk(scores, k, dim=-1, sorted=False)
        weights    = torch.sigmoid(scores.gather(1, top_idx))    # (B, k)

        # Sort by position — causal SSM correctness requirement
        top_idx_sorted, perm = top_idx.sort(dim=-1)
        weights_sorted       = weights.gather(1, perm)

        idx_exp   = top_idx_sorted.unsqueeze(-1).expand(B, k, D)
        x_sel     = x.gather(1, idx_exp)                         # (B, k, D)
        block_out = self.block(x_sel)                            # (B, k, D)

        # MoD residual: selected positions get x + weight · block(x)
        x_proc = x_sel + weights_sorted.unsqueeze(-1) * block_out

        out = x.clone()
        out.scatter_(1, idx_exp, x_proc)
        return out


class AgentAttnBlock(nn.Module):
    """
    "Agent" attention block: Dynamic Sparse GQA + Fine-Grained MoE.

    The Dynamic Sparse GQA provides head-level sparsity (fewer active heads
    per token) while the MoE provides expert-level sparsity (fewer active
    experts per token). Both are differentiable through their gate/weight values.

    Residuals live inside this block. ContinuousMemory (upstream) handles
    cross-token persistent fact retrieval; this block handles in-context
    reasoning and conditional knowledge routing.
    """

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn      = DynamicSparseGQA(cfg)
        self.ffn_norm  = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.moe       = FineGrainedMoE(cfg)

    def forward(
        self,
        x: torch.Tensor,
        position_ids: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        x = x + self.attn(self.attn_norm(x), position_ids)
        moe_out, load = self.moe(self.ffn_norm(x))
        x = x + moe_out
        return x, load


class ADA(nn.Module):
    """
    ADA v4.5 — Adaptive Depth Architecture

    32-layer topology (block_pattern property):
      Layer  0, 1  : MoD( Mamba2Block )    linear-complexity selective SSM
      Layer  2     : ContinuousMemory      external learnable KV retrieval
      Layer  3     : AgentAttnBlock        dynamic sparse GQA + fine-grained MoE
      … × 8 cycles

    Loss:
      L = L_CE(t+1) + 0.3 × mean_i( L_CE(t+i+2) )
      No auxiliary MoE loss. expert_bias absorbs load imbalance signal.
    """

    def __init__(self, cfg: ADAConfig):
        super().__init__()
        self.cfg = cfg

        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop  = nn.Dropout(cfg.dropout)

        self.layers: nn.ModuleList = nn.ModuleList()
        for bt in cfg.block_pattern:
            if bt == "mamba":
                self.layers.append(
                    MoD(Mamba2Block(cfg), cfg.d_model, cfg.mod_capacity)
                )
            elif bt == "memory":
                self.layers.append(ContinuousMemory(cfg))
            elif bt == "agent_attn":
                self.layers.append(AgentAttnBlock(cfg))
            else:
                raise ValueError(f"Unknown block type: {bt!r}")

        self.norm    = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight   # weight tying

        # MTP: head i predicts token at position t + i + 2
        self.mtp_heads = nn.ModuleList([
            nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            for _ in range(cfg.n_mtp)
        ])

        self._init_weights()

    def _init_weights(self):
        std = 0.02
        nn.init.trunc_normal_(self.embed.weight, std=std)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=std)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.trunc_normal_(m.weight, std=std)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, RMSNorm):
                nn.init.ones_(m.weight)

    def forward(
        self,
        input_ids: torch.Tensor,
        position_ids: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, List[torch.Tensor], List[torch.Tensor]]:
        """
        Parameters
        ----------
        input_ids   : (B, T)
        position_ids: (B, T) or None

        Returns
        -------
        logits       : (B, T, vocab_size)
        mtp_logits   : list of n_mtp tensors (B, T, vocab_size)
        expert_loads : list of (n_experts,) per AgentAttn layer
        """
        B, T = input_ids.shape
        if position_ids is None:
            position_ids = (
                torch.arange(T, device=input_ids.device)
                .unsqueeze(0).expand(B, -1)
            )

        x = self.drop(self.embed(input_ids))

        expert_loads: List[torch.Tensor] = []
        for layer in self.layers:
            if isinstance(layer, AgentAttnBlock):
                x, load = layer(x, position_ids)
                expert_loads.append(load)
            else:  # MoD(Mamba2Block) or ContinuousMemory
                x = layer(x)

        x      = self.norm(x)
        logits = self.lm_head(x)
        mtp_logits = [head(x) for head in self.mtp_heads]

        return logits, mtp_logits, expert_loads

    def compute_loss(
        self,
        input_ids: torch.Tensor,
        position_ids: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        L = L_main + 0.3 × mean(L_mtp)
        L_main: next-token CE (t → t+1)
        L_mtp : head i: hidden at t → token at t+i+2
        """
        B, T = input_ids.shape
        V    = self.cfg.vocab_size

        logits, mtp_logits, expert_loads = self.forward(input_ids, position_ids)

        main_loss = F.cross_entropy(
            logits[:, :-1].reshape(-1, V),
            input_ids[:, 1:].reshape(-1),
        )

        mtp_loss = torch.tensor(0.0, device=input_ids.device)
        n_valid  = 0
        for i, hl in enumerate(mtp_logits):
            offset = i + 2
            if T - offset <= 0:
                continue
            mtp_loss = mtp_loss + F.cross_entropy(
                hl[:, : T - offset].reshape(-1, V),
                input_ids[:, offset:].reshape(-1),
            )
            n_valid += 1
        if n_valid:
            mtp_loss = mtp_loss / n_valid

        total = main_loss + 0.3 * mtp_loss

        load_ent = 0.0
        if expert_loads:
            for ld in expert_loads:
                p = ld.float() / (ld.sum() + 1e-9)
                load_ent += -(p * (p + 1e-9).log()).sum().item()
            load_ent /= len(expert_loads)

        return total, {
            "loss":         total.item(),
            "loss_main":    main_loss.item(),
            "loss_mtp":     mtp_loss.item(),
            "load_entropy": load_ent,
        }

## Paired-sequence correctness fixture

Two sequences share their first four token IDs. A causal model must
assign identical logits at those positions regardless of the suffix.
First isolate the head gate using its own inputs. Then measure the
original assembled model, where both the head gate and depth selection
can depend on the suffix. The negative control is expected to fail.

In [3]:
config = ADAConfig(
    vocab_size=16, d_model=24, n_heads=4, n_kv_heads=2, n_layers=4,
    mamba_d_state=4, mamba_expand=1, mamba_n_heads=4, mamba_d_conv=3,
    mem_size=8, d_mem=6, sparse_keep=0.5, n_experts=4,
    n_active=2, moe_d_ff=24, n_mtp=2, max_seq_len=16,
)
historical = ADA(config).eval()
base_ids = torch.tensor([[0, 1, 2, 3, 4, 5, 6, 7]])
changed_ids = torch.tensor([[0, 1, 2, 3, 14, 12, 10, 8]])
cutoff = 4

with torch.no_grad():
    historical_attention = historical.layers[-1].attn
    component_delta = (
        historical_attention(historical.embed(base_ids))[:, :cutoff]
        - historical_attention(historical.embed(changed_ids))[:, :cutoff]
    ).abs().max().item()
    historical_delta = (
        historical(base_ids)[0][:, :cutoff] - historical(changed_ids)[0][:, :cutoff]
    ).abs().max().item()

tolerance = 2e-6
assert component_delta > tolerance, "negative control did not expose the head gate"
assert historical_delta > tolerance, "negative control did not expose full-model leakage"
print(f"Historical head-gate prefix delta: {component_delta:.8f} | FAIL (expected)")
print(f"Historical model prefix delta:     {historical_delta:.8f} | FAIL (expected)")

Historical head-gate prefix delta: 0.00010578 | FAIL (expected)


Historical model prefix delta:     0.04083619 | FAIL (expected)


## Causal variant

Both changes are necessary to isolate a causal training path here.
Prefix pooling makes each gate a function of the observed prefix.
The full-depth wrapper preserves the original SSM's contiguous timeline
and reuses the original depth-router weights as a per-token residual gate.
It deliberately executes all tokens. This is a correctness reference
for rebuilding adaptive compute, with no claim of reduced FLOPs.

In [4]:
class PrefixGatedGQA(DynamicSparseGQA):
    def forward(
        self,
        x: torch.Tensor,
        position_ids: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        B, T, _ = x.shape
        hd = self.head_dim

        q = self.wq(x).reshape(B, T, self.n_heads, hd).transpose(1, 2)  # (B,H,T,hd)
        k = self.wk(x).reshape(B, T, self.n_kv,   hd).transpose(1, 2)
        v = self.wv(x).reshape(B, T, self.n_kv,   hd).transpose(1, 2)

        q = apply_rope(q, self.rope_cos, self.rope_sin, position_ids)
        k = apply_rope(k, self.rope_cos, self.rope_sin, position_ids)

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # A gate at t may depend on tokens through t, including x[t].
        prefix_length = torch.arange(1, T + 1, device=x.device, dtype=x.dtype)
        prefix_mean = x.cumsum(dim=1) / prefix_length.view(1, T, 1)
        gate_scores = torch.sigmoid(self.head_gate_proj(prefix_mean))
        topk_vals, topk_idx = torch.topk(gate_scores, self.k_keep, dim=-1)
        head_mask = torch.zeros_like(gate_scores).scatter(-1, topk_idx, topk_vals)
        head_mask = head_mask.transpose(1, 2).unsqueeze(-1)

        # ── standard scaled dot-product attention ─────────────────────────────
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale       # (B,H,T,T)
        causal = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~causal[None, None], float("-inf"))

        attn = F.softmax(scores.float(), dim=-1).to(x.dtype)
        attn = self.drop(attn)

        # Apply head mask: zero out pruned heads; weight active heads by gate.
        out = (torch.matmul(attn, v) * head_mask)                        # (B,H,T,hd)
        out = out.transpose(1, 2).reshape(B, T, -1)
        return self.wo(out)

In [5]:
class FullDepthSSM(nn.Module):
    def __init__(self, historical_wrapper):
        super().__init__()
        self.block = historical_wrapper.block
        self.router = historical_wrapper.router

    def forward(self, x):
        return x + torch.sigmoid(self.router(x)) * self.block(x)


def create_causal_model(config):
    model = ADA(config)
    for index, layer in enumerate(model.layers):
        if isinstance(layer, MoD):
            model.layers[index] = FullDepthSSM(layer)
        elif isinstance(layer, AgentAttnBlock):
            causal_attention = PrefixGatedGQA(config)
            causal_attention.load_state_dict(layer.attn.state_dict())
            layer.attn = causal_attention
    return model


model = create_causal_model(config)
model.load_state_dict(historical.state_dict())


@torch.no_grad()
def check_causal_prefix(model):
    model.eval()
    first = model(base_ids)[0][:, :cutoff]
    second = model(changed_ids)[0][:, :cutoff]
    shortened = model(base_ids[:, :cutoff])[0]
    torch.testing.assert_close(first, second, rtol=0, atol=tolerance)
    torch.testing.assert_close(first, shortened, rtol=0, atol=tolerance)
    return (first - second).abs().max().item()


print(f"Causal variant before training: prefix delta = {check_causal_prefix(model):.8f}")

Causal variant before training: prefix delta = 0.00000000


## Synthetic training and held-out starts

Each token follows its predecessor modulo 16. Training and evaluation
use disjoint starting IDs, while the transition rule is shared. Optimize
the historical next-token plus multi-token objective in the corrected
model. Read the held-out next-token loss separately from the joint loss.

In [6]:
def cyclic_examples(starts, length, vocab):
    return (torch.as_tensor(starts)[:, None] + torch.arange(length)[None, :]) % vocab


def next_token_metrics(logits, ids):
    prediction = logits[:, :-1]
    labels = ids[:, 1:]
    loss = F.cross_entropy(prediction.flatten(0, 1), labels.flatten())
    accuracy = (prediction.argmax(-1) == labels).float().mean()
    return loss, accuracy

In [7]:
train_ids = cyclic_examples(range(12), 10, config.vocab_size)
eval_ids = cyclic_examples(range(12, 16), 10, config.vocab_size)
assert torch.unique(torch.cat((train_ids, eval_ids)), dim=0).shape[0] == 16


@torch.no_grad()
def evaluate(model):
    model.eval()
    return tuple(float(value) for value in next_token_metrics(model(eval_ids)[0], eval_ids))


initial_loss, initial_accuracy = evaluate(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.015)
model.train()
for step in range(32):
    optimizer.zero_grad(set_to_none=True)
    loss, components = model.compute_loss(train_ids)
    assert torch.isfinite(loss)
    loss.backward()
    gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    assert torch.isfinite(gradient_norm) and gradient_norm > 0
    optimizer.step()
    if (step + 1) % 8 == 0:
        print(f"step {step + 1:2d}: next {components['loss_main']:.4f}, MTP {components['loss_mtp']:.4f}")

final_loss, final_accuracy = evaluate(model)
assert final_loss < initial_loss
print(f"Held-out next-token CE: {initial_loss:.6f} -> {final_loss:.6f}")
print(f"Held-out accuracy: {initial_accuracy:.1%} -> {final_accuracy:.1%}")
print(f"Causal variant after training: prefix delta = {check_causal_prefix(model):.8f}")

step  8: next 1.1572, MTP 1.2838


step 16: next 0.2370, MTP 0.2265


step 24: next 0.0321, MTP 0.0284


step 32: next 0.0073, MTP 0.0063
Held-out next-token CE: 2.759652 -> 0.013169
Held-out accuracy: 0.0% -> 100.0%
Causal variant after training: prefix delta = 0.00000000


## Greedy generation and artifact round trip

Sequential greedy decoding recomputes the prefix each step. A
tensor-only state-dict round trip checks bundle reproducibility for this
tiny configuration. No remote or third-party checkpoints are loaded.

In [8]:
@torch.no_grad()
def greedy_decode(model, prompt, new_tokens):
    model.eval()
    ids = prompt.clone()
    for _ in range(new_tokens):
        ids = torch.cat((ids, model(ids)[0][:, -1].argmax(-1, keepdim=True)), dim=1)
    return ids


generated = greedy_decode(model, torch.tensor([[13, 14]]), 6)
assert generated.shape == (1, 8)
checkpoint = BytesIO()
torch.save(model.state_dict(), checkpoint)
checkpoint.seek(0)
copy = create_causal_model(config).eval()
copy.load_state_dict(torch.load(checkpoint, weights_only=True))
with torch.no_grad():
    torch.testing.assert_close(copy(eval_ids)[0], model(eval_ids)[0], rtol=0, atol=0)
print("Generated IDs:", generated.tolist())
print("Tensor-only checkpoint round trip: PASS")

Generated IDs: [[13, 14, 15, 0, 1, 2, 3, 4]]
Tensor-only checkpoint round trip: PASS


## Decision

Preserve the historical model as a negative control. Extend future
experiments with separately specified causal depth scheduling, larger
sequence fixtures, and dense versus sparse hardware measurements.
Synthetic fitting and this finite suite provide a bounded check of
implementation behavior.